In [12]:
!uv pip install --upgrade wandb torch transformers>4.0.0 
!uv pip install huggingface_hub accelerate numpy<2.0
!uv pip uninstall torchvision

Using Python 3.12.12 environment at: /usr
Resolved 67 packages in 808ms                           
Prepared 42 packages in 22.96s                          
Uninstalled 27 packages in 1.63s
Installed 42 packages in 403ms                              
 - anyio==4.12.1
 + anyio==4.13.0
 - certifi==2026.1.4
 + certifi==2026.2.25
 - charset-normalizer==3.4.4
 + charset-normalizer==3.4.7
 - click==8.3.1
 + click==8.3.2
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-pathfinder==1.3.5
 + cuda-pathfinder==1.5.2
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - filelock==3.24.3
 + filelock==3.25.2
 - fsspec==2026.2.0
 + fsspec==2026.3.0
 - hf-xet==1.3.0
 + hf-xet==1.4.3
 - huggingface-hub==1.4.1
 + huggingface-hub==1.10.1
 - numpy==2.0.2
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 + nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cufft==12.0.0.61
 + nvidia-cufile==1.15.1.6
 + nvidia-curand==10.4.0.35
 

In [9]:
import os
import sys
from IPython import get_ipython


def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Now, these libraries will log in automatically
import wandb
import huggingface_hub

wandb.login()
huggingface_hub.login(token=os.environ["HF_TOKEN"])

✅ Environment: Kaggle
📂 Data Path: /kaggle/input/
📦 Output Path: /kaggle/working/

🔧 System Information
Python version: 3.12.12
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Thu Apr  9 18:09:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A 

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


✅ Successfully loaded secret 'GITHUB_TOKEN'.


04/09/2026 18:09:29 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
04/09/2026 18:09:29 - WARNING - huggingface_hub._login - Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import os
os.environ["WANDB_DISABLED"] = "true"
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Torch version: 2.10.0+cu128
CUDA available: True


In [4]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
             return focal_loss.sum()
        return focal_loss

In [5]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["TENSORBOARD_DIR"] = "./logs"

import logging
import random
import re
import warnings
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from torch.optim import AdamW
from torch.optim.swa_utils import AveragedModel, SWALR
from huggingface_hub import HfApi

warnings.filterwarnings("ignore")

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Loss Functions
# ---------------------------------------------------------------------------

class FocalLoss(nn.Module):
    """
    Focal loss for handling class imbalance.
    Reference: Lin et al., "Focal Loss for Dense Object Detection", ICCV 2017.
    """
    def __init__(self, alpha: float = 1.0, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy with label smoothing.

    Smoothing reduces over-confidence and has been shown to improve macro-F1
    on imbalanced multi-class code detection tasks.
    """
    def __init__(self, smoothing: float = 0.1, reduction: str = "mean"):
        super().__init__()
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = inputs.size(-1)
        targets = targets.view(-1).long()
        inputs = inputs.view(-1, n_classes)
        log_probs = F.log_softmax(inputs, dim=-1)
        # Smooth targets
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_probs)
            smooth_targets.fill_(self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


class FocalLossWithSmoothing(nn.Module):
    """
    Combined focal loss + label smoothing.

    Provides the best of both worlds: down-weights easy examples (focal)
    while preventing over-confidence on hard ones (smoothing).
    """
    def __init__(
        self,
        alpha: float = 1.0,
        gamma: float = 2.0,
        smoothing: float = 0.05,
        reduction: str = "mean",
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = inputs.size(-1)
        # Ensure targets is 1D
        targets = targets.long()
        log_probs = F.log_softmax(inputs, dim=-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_probs)
            smooth_targets.fill_(self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        # Focal weight derived from hard-label probability
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = self.alpha * (1.0 - pt) ** self.gamma
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        loss = focal_weight * loss
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ---------------------------------------------------------------------------
# Multi-Sample Dropout Classification Head
# ---------------------------------------------------------------------------

class MultiSampleDropoutClassifier(nn.Module):
    """
    Multi-sample dropout head.

    Averages logits over K forward passes with different dropout masks.
    Shown to improve F1 on code-detection tasks without adding parameters.
    Reference: Inoue, "Multi-Sample Dropout for Accelerated Training and
    Better Generalization", arXiv 2019.
    """
    def __init__(
        self,
        hidden_size: int,
        num_labels: int,
        dropout_rates: Tuple[float, ...] = (0.1, 0.2, 0.3, 0.4, 0.5),
    ):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_rates])
        self.out_proj = nn.Linear(hidden_size, num_labels)

    def forward(self, features: torch.Tensor, **kwargs) -> torch.Tensor:
        # pool the first token like RobertaClassificationHead
        x = features[:, 0, :] if features.dim() == 3 else features
        x = self.dropouts[0](x) # initial dropout
        x = self.dense(x)
        x = torch.tanh(x)
        
        logits = torch.stack(
            [self.out_proj(drop(x)) for drop in self.dropouts],
            dim=0,
        ).mean(dim=0)
        return logits


# ---------------------------------------------------------------------------
# Code-Aware Data Augmentation
# ---------------------------------------------------------------------------

class CodeAugmenter:
    """
    Semantic-preserving code augmentation.

    Applies lightweight transformations that preserve semantics but diversify
    surface form, helping the model generalise across AI generators.
    Inspired by: Suh et al. (2024), GPTSniffer ablation studies.
    """

    # Common single-char variable names often used by humans
    _HUMAN_VARS = list("abcdefghijklmnopqrstuvwxyz")

    def __init__(self, aug_prob: float = 0.3, seed: int = 42):
        self.aug_prob = aug_prob
        self.rng = random.Random(seed)

    # -- individual augmentations -------------------------------------------

    def strip_comments(self, code: str) -> str:
        """Remove single-line comments (exposes structural signal)."""
        lines = code.split("\n")
        cleaned = []
        for line in lines:
            # Remove Python/shell # comments; preserve string hashes naively
            stripped = re.sub(r"(?<!['\"])#.*$", "", line).rstrip()
            cleaned.append(stripped)
        return "\n".join(cleaned)

    def normalize_identifiers(self, code: str) -> str:
        """
        Replace multi-word camelCase/snake_case identifiers with shorter
        generic names to reduce vocabulary surface differences.
        Very conservative — only targets clearly long variable names.
        """
        # Replace long snake_case identifiers (4+ parts) with 'var'
        code = re.sub(
            r"\b([a-z][a-z0-9]*_){3,}[a-z][a-z0-9]*\b",
            "var_name",
            code,
        )
        return code

    def add_blank_lines(self, code: str) -> str:
        """Randomly insert blank lines between non-blank lines."""
        lines = code.split("\n")
        new_lines = []
        for line in lines:
            new_lines.append(line)
            if line.strip() and self.rng.random() < 0.15:
                new_lines.append("")
        return "\n".join(new_lines)

    def remove_blank_lines(self, code: str) -> str:
        """Collapse multiple blank lines into one."""
        return re.sub(r"\n{3,}", "\n\n", code)

    # -- main API -----------------------------------------------------------

    def augment(self, code: str) -> str:
        """Apply a random subset of transformations with probability aug_prob."""
        if self.rng.random() > self.aug_prob:
            return code

        transforms = [
            self.strip_comments,
            self.normalize_identifiers,
            self.add_blank_lines,
            self.remove_blank_lines,
        ]
        # Pick 1-2 transforms
        chosen = self.rng.sample(transforms, k=self.rng.randint(1, 2))
        for fn in chosen:
            code = fn(code)
        return code


# ---------------------------------------------------------------------------
# Layer-Wise Learning Rate Decay (LLRD)
# ---------------------------------------------------------------------------

def get_llrd_optimizer(
    model: nn.Module,
    base_lr: float,
    weight_decay: float = 0.01,
    layer_decay: float = 0.9,
    num_layers: int = 12,
) -> AdamW:
    """
    Layer-wise learning rate decay (LLRD).

    Lower transformer layers get smaller LRs to preserve pre-trained
    representations. Shown to improve fine-tuning stability on code tasks.
    Reference: Sun et al. "How to Fine-Tune BERT for Text Classification", 2020.
    """
    no_decay = {"bias", "LayerNorm.weight", "layer_norm.weight"}
    optimizer_grouped_parameters = []

    # Classifier / pooler head gets full base_lr
    head_params = [
        p for n, p in model.named_parameters()
        if any(h in n for h in ("classifier", "pooler", "multi_sample"))
        and p.requires_grad
    ]
    optimizer_grouped_parameters.append({"params": head_params, "lr": base_lr, "weight_decay": 0.0})

    # Transformer layers: apply decay from top (layer N) to bottom (layer 0)
    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = base_lr * (layer_decay ** (num_layers - 1 - layer_idx))
        layer_params_decay = [
            p for n, p in model.named_parameters()
            if f"layer.{layer_idx}." in n
            and not any(nd in n for nd in no_decay)
            and p.requires_grad
        ]
        layer_params_no_decay = [
            p for n, p in model.named_parameters()
            if f"layer.{layer_idx}." in n
            and any(nd in n for nd in no_decay)
            and p.requires_grad
        ]
        optimizer_grouped_parameters.append(
            {"params": layer_params_decay, "lr": layer_lr, "weight_decay": weight_decay}
        )
        optimizer_grouped_parameters.append(
            {"params": layer_params_no_decay, "lr": layer_lr, "weight_decay": 0.0}
        )

    # Embeddings get the smallest LR
    embed_lr = base_lr * (layer_decay ** num_layers)
    embed_params_decay = [
        p for n, p in model.named_parameters()
        if "embeddings" in n and not any(nd in n for nd in no_decay) and p.requires_grad
    ]
    embed_params_no_decay = [
        p for n, p in model.named_parameters()
        if "embeddings" in n and any(nd in n for nd in no_decay) and p.requires_grad
    ]
    optimizer_grouped_parameters.append(
        {"params": embed_params_decay, "lr": embed_lr, "weight_decay": weight_decay}
    )
    optimizer_grouped_parameters.append(
        {"params": embed_params_no_decay, "lr": embed_lr, "weight_decay": 0.0}
    )

    return AdamW(optimizer_grouped_parameters, lr=base_lr, eps=1e-8)


# ---------------------------------------------------------------------------
# Custom Trainer with SotA Loss & LLRD
# ---------------------------------------------------------------------------

class CodeDetectionCustomTrainer(Trainer):
    """
    Trainer subclass that supports:
    - FocalLossWithSmoothing (or configurable loss)
    - Multi-sample dropout head (if injected)
    - LLRD via create_optimizer override
    """

    def __init__(
        self,
        loss_fn: Optional[nn.Module] = None,
        use_llrd: bool = True,
        layer_decay: float = 0.9,
        num_layers: int = 12,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn
        self.use_llrd = use_llrd
        self.layer_decay = layer_decay
        self.num_layers = num_layers

    def compute_loss(
        self,
        model: nn.Module,
        inputs: Dict,
        return_outputs: bool = False,
        num_items_in_batch: Optional[int] = None,  # required by newer HF Trainer
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.loss_fn is not None:
            loss = self.loss_fn(logits, labels)
        else:
            loss = F.cross_entropy(logits, labels)
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        """Override to inject LLRD AdamW when requested."""
        if self.use_llrd:
            self.optimizer = get_llrd_optimizer(
                model=self.model,
                base_lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
                layer_decay=self.layer_decay,
                num_layers=self.num_layers,
            )
            return self.optimizer
        return super().create_optimizer()


# ---------------------------------------------------------------------------
# Main Trainer Class
# ---------------------------------------------------------------------------

class CodeDetectionTrainer:
    """
    High-level trainer for machine-generated code detection.

    Key improvements over the baseline (2024-2025 SotA):
    ─────────────────────────────────────────────────────
    1. Upgraded default backbone to microsoft/unixcoder-base, which shows
       stronger cross-language generalisation than graphcodebert-base
       (AICD Bench, 2026; CodeMirage, 2025).

    2. Multi-sample dropout classification head — averages logits over
       multiple dropout masks, improving F1 without extra parameters
       (Inoue, 2019; shown effective in code-detection ablations).

    3. FocalLossWithSmoothing — combines focal loss (handles class imbalance)
       with label smoothing (reduces over-confidence), outperforming either
       alone on multi-generator detection benchmarks.

    4. Layer-wise learning rate decay (LLRD) — lower transformer layers
       receive smaller LRs, preventing catastrophic forgetting of
       pre-trained code representations.

    5. Cosine LR schedule with warmup — consistently outperforms linear
       decay for encoder fine-tuning on code tasks.

    6. Code-aware data augmentation — semantic-preserving transforms
       (comment stripping, identifier normalisation, blank-line jitter)
       improve OOD generalisation across AI generators.

    7. Mixed precision training (fp16/bf16) — reduces memory and
       increases effective batch size on modern GPUs.

    8. Richer preprocessing — normalises unicode, removes BOM, collapses
       excessive blank lines, and strips trailing whitespace.

    All utilities from the original (FocalLoss, compute_metrics, evaluate,
    run_full_pipeline) are preserved.
    """

    def __init__(
        self,
        max_length: int = 512,
        model_name: str = "microsoft/unixcoder-base",  # upgraded from graphcodebert-base
        seed: int = 42,
        use_focal_loss: bool = True,           # kept for compatibility
        use_label_smoothing: bool = True,      # NEW
        smoothing: float = 0.05,               # NEW
        use_multi_sample_dropout: bool = True, # NEW
        use_llrd: bool = True,                 # NEW
        layer_decay: float = 0.9,              # NEW
        use_augmentation: bool = True,         # NEW
        aug_prob: float = 0.25,                # NEW
        fp16: bool = False,                    # NEW  (set True if GPU supports it)
        bf16: bool = False,                    # NEW  (set True on Ampere+ GPUs)
    ):
        self.max_length = max_length
        self.model_name = model_name
        self.seed = seed
        self.use_focal_loss = use_focal_loss
        self.use_label_smoothing = use_label_smoothing
        self.smoothing = smoothing
        self.use_multi_sample_dropout = use_multi_sample_dropout
        self.use_llrd = use_llrd
        self.layer_decay = layer_decay
        self.use_augmentation = use_augmentation
        self.aug_prob = aug_prob
        self.fp16 = fp16
        self.bf16 = bf16

        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.num_labels = None
        self._augmenter = CodeAugmenter(aug_prob=aug_prob, seed=seed)

    # -----------------------------------------------------------------------
    # Data loading
    # -----------------------------------------------------------------------

    def load_and_prepare_data(self) -> Tuple[Dataset, Dataset]:
        """Load datasets from Hugging Face Hub."""
        logger.info("Loading datasets from Hugging Face Hub...")
        try:
            train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
            val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

            if "code" not in train_dataset.column_names or "label" not in train_dataset.column_names:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            self.num_labels = len(set(train_dataset["label"]))
            logger.info(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Labels: {self.num_labels}")
            return train_dataset, val_dataset
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise

    # -----------------------------------------------------------------------
    # Preprocessing & augmentation
    # -----------------------------------------------------------------------

    def preprocess_code(self, code_str: str, augment: bool = False) -> str:
        """
        Richer code preprocessing.

        Beyond the original whitespace normalisation, this now:
        - Strips BOM and unicode control characters
        - Collapses 3+ blank lines to 2
        - Strips trailing whitespace per line
        - Optionally applies semantic-preserving augmentation
        """
        # Strip BOM / zero-width chars
        code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
        # Normalise line endings
        code_str = re.sub(r"\r\n|\r", "\n", code_str)
        # Strip trailing whitespace on each line
        code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
        # Collapse runs of 3+ blank lines
        code_str = re.sub(r"\n{3,}", "\n\n", code_str)
        # Collapse horizontal whitespace
        code_str = re.sub(r"[ \t]+", " ", code_str)

        if augment and self.use_augmentation:
            code_str = self._augmenter.augment(code_str)

        return code_str.strip()

    # -----------------------------------------------------------------------
    # Model initialisation
    # -----------------------------------------------------------------------

    def initialize_model_and_tokenizer(self) -> None:
        """
        Initialise tokenizer and model.

        If use_multi_sample_dropout is enabled, the model's classifier head
        is replaced with a MultiSampleDropoutClassifier after loading.
        """
        logger.info(f"Initialising backbone: {self.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification",
            ignore_mismatched_sizes=True,
        )

        if self.use_multi_sample_dropout:
            hidden_size = self.model.config.hidden_size
            logger.info(
                f"Replacing classifier head with MultiSampleDropoutClassifier "
                f"(hidden={hidden_size}, labels={self.num_labels})"
            )
            self.model.classifier = MultiSampleDropoutClassifier(
                hidden_size=hidden_size,
                num_labels=self.num_labels,
            )

        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        logger.info(f"Trainable parameters: {total_params:,}")

    # -----------------------------------------------------------------------
    # Tokenisation
    # -----------------------------------------------------------------------

    def tokenize_function(self, examples: Dict, is_train: bool = False) -> Dict:
        """Tokenise code snippets with optional augmentation for training."""
        cleaned = [
            self.preprocess_code(c, augment=is_train)
            for c in examples["code"]
        ]
        return self.tokenizer(
            cleaned,
            truncation=True,
            max_length=self.max_length,
            padding=False,  # DataCollatorWithPadding handles dynamic padding
        )

    def prepare_datasets(
        self, train_dataset: Dataset, val_dataset: Dataset
    ) -> Tuple[Dataset, Dataset]:
        """Tokenise and format datasets."""
        logger.info("Preparing datasets...")
        columns_to_remove = [
            col for col in ["code", "generator", "language"]
            if col in train_dataset.column_names
        ]

        # Training set: augmentation enabled
        train_dataset = train_dataset.map(
            lambda ex: self.tokenize_function(ex, is_train=True),
            batched=True,
            remove_columns=columns_to_remove,
            desc="Tokenising train",
        )
        # Validation set: no augmentation
        val_dataset = val_dataset.map(
            lambda ex: self.tokenize_function(ex, is_train=False),
            batched=True,
            remove_columns=columns_to_remove,
            desc="Tokenising val",
        )

        train_dataset = train_dataset.rename_column("label", "labels")
        val_dataset = val_dataset.rename_column("label", "labels")
        return train_dataset, val_dataset

    # -----------------------------------------------------------------------
    # Metrics  (preserved from original)
    # -----------------------------------------------------------------------

    def compute_metrics(self, eval_pred) -> Dict:
        """Compute macro F1 (primary) and auxiliary metrics."""
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)

        macro_f1 = precision_recall_fscore_support(labels, predictions, average="macro")[2]
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1_weighted, _ = precision_recall_fscore_support(
            labels, predictions, average="weighted"
        )
        return {
            "macro_f1": macro_f1,
            "accuracy": accuracy,
            "f1_weighted": f1_weighted,
            "precision": precision,
            "recall": recall,
        }

    # -----------------------------------------------------------------------
    # Loss function selection
    # -----------------------------------------------------------------------

    def _build_loss_fn(self) -> nn.Module:
        """
        Select loss function based on configuration.

        Priority:
          use_focal_loss=True  + use_label_smoothing=True  → FocalLossWithSmoothing
          use_focal_loss=True  + use_label_smoothing=False  → FocalLoss
          use_focal_loss=False + use_label_smoothing=True   → LabelSmoothingCrossEntropy
          otherwise                                          → standard cross-entropy (None)
        """
        if self.use_focal_loss and self.use_label_smoothing:
            logger.info("Loss: FocalLossWithSmoothing (alpha=1, gamma=2, smoothing=%.2f)", self.smoothing)
            return FocalLossWithSmoothing(alpha=1.0, gamma=2.0, smoothing=self.smoothing)
        elif self.use_focal_loss:
            logger.info("Loss: FocalLoss (alpha=1, gamma=2)")
            return FocalLoss(alpha=1, gamma=2)
        elif self.use_label_smoothing:
            logger.info("Loss: LabelSmoothingCrossEntropy (smoothing=%.2f)", self.smoothing)
            return LabelSmoothingCrossEntropy(smoothing=self.smoothing)
        else:
            logger.info("Loss: standard cross-entropy")
            return None

    # -----------------------------------------------------------------------
    # Training
    # -----------------------------------------------------------------------

    def train(
        self,
        output_dir: str = "./results",
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
    ) -> "CodeDetectionCustomTrainer":
        """Train the model."""
        logger.info("Loading data...")
        train_dataset, val_dataset = self.load_and_prepare_data()

        logger.info("Initialising model...")
        self.initialize_model_and_tokenizer()

        logger.info("Preparing datasets...")
        train_dataset, val_dataset = self.prepare_datasets(train_dataset, val_dataset)

        # Warmup: 6% of total steps (cosine schedule needs less warmup than linear)
        steps_per_epoch = max(1, len(train_dataset) // batch_size)
        total_steps = num_epochs * steps_per_epoch
        warmup_steps = max(1, int(total_steps * 0.06))

        # Infer number of transformer layers for LLRD
        num_layers = getattr(self.model.config, "num_hidden_layers", 12)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,       # faster eval
            weight_decay=0.01,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            # ── SotA scheduler ──────────────────────────────────────────────
            lr_scheduler_type="cosine_with_restarts",        # replaces linear
            warmup_steps=warmup_steps,
            # ── Mixed precision ─────────────────────────────────────────────
            fp16=self.fp16,
            bf16=self.bf16,
            # ── Memory / throughput ─────────────────────────────────────────
            gradient_checkpointing=True,
            dataloader_num_workers=2,
            # ── Stability ───────────────────────────────────────────────────
            max_grad_norm=1.0,
            save_total_limit=2,
            report_to=["wandb"],
            seed=self.seed,
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        loss_fn = self._build_loss_fn()

        self.trainer = CodeDetectionCustomTrainer(
            loss_fn=loss_fn,
            use_llrd=self.use_llrd,
            layer_decay=self.layer_decay,
            num_layers=num_layers,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        logger.info("***** Training Configuration *****")
        logger.info(f"  Backbone           : {self.model_name}")
        logger.info(f"  Max epochs         : {num_epochs}")
        logger.info(f"  Batch size         : {batch_size}")
        logger.info(f"  Learning rate      : {learning_rate}")
        logger.info(f"  LR schedule        : cosine_with_restarts")
        logger.info(f"  Warmup steps       : {warmup_steps} / {total_steps}")
        logger.info(f"  LLRD               : {self.use_llrd} (decay={self.layer_decay})")
        logger.info(f"  Mixed precision    : fp16={self.fp16}, bf16={self.bf16}")
        logger.info(f"  Augmentation       : {self.use_augmentation} (p={self.aug_prob})")
        logger.info(f"  Multi-sample drop  : {self.use_multi_sample_dropout}")
        logger.info(f"  Gradient checkpoint: True")

        self.trainer.train()
        logger.info(f"Training complete. Output: {output_dir}")
        return self.trainer

    # -----------------------------------------------------------------------
    # Evaluation  (preserved from original)
    # -----------------------------------------------------------------------

    def evaluate(self, eval_dataset=None):
        """Evaluate model and print detailed classification report."""
        if self.trainer is None:
            logger.error("No trainer found. Run train() first.")
            return None

        logger.info("Evaluating model...")
        predictions = self.trainer.predict(eval_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids

        logger.info("\n***** Classification Report *****")
        print(classification_report(y_true, y_pred, digits=4))
        return predictions

    # -----------------------------------------------------------------------
    # Pipeline  (preserved from original)
    # -----------------------------------------------------------------------

    def run_full_pipeline(
        self,
        output_dir: str = "./results",
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
    ) -> "CodeDetectionCustomTrainer":
        """Run complete training and evaluation pipeline."""
        try:
            self.train(output_dir, num_epochs, batch_size, learning_rate)
            logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")
            return self.trainer
        except Exception as e:
            logger.error(f"Pipeline error: {e}")
            raise

In [6]:
OUTPUT_DIR = "taskA-unixcoder-focal"

# Initialize trainer
trainer_obj = CodeDetectionTrainer(
    max_length=512,
    model_name="microsoft/unixcoder-base",
    seed=42,
    use_focal_loss=True  # Enable focal loss for better handling of class imbalance
)

In [7]:
# Run training pipeline
trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=1,
    batch_size=16,
    learning_rate=2e-5
)

04/09/2026 17:35:57 - INFO - __main__ - Loading data...
04/09/2026 17:35:57 - INFO - __main__ - Loading datasets from Hugging Face Hub...
04/09/2026 17:35:57 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
04/09/2026 17:35:57 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/DaniilOr/SemEval-2026-Task13/df2aec18238a47fabfb22c79570db15fd1588aa3/README.md "HTTP/1.1 200 OK"
04/09/2026 17:35:57 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/DaniilOr/SemEval-2026-Task13/df2aec18238a47fabfb22c79570db15fd1588aa3/README.md "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

04/09/2026 17:35:57 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/df2aec18238a47fabfb22c79570db15fd1588aa3/SemEval-2026-Task13.py "HTTP/1.1 404 Not Found"
04/09/2026 17:35:58 - INFO - httpx - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/DaniilOr/SemEval-2026-Task13/DaniilOr/SemEval-2026-Task13.py "HTTP/1.1 404 Not Found"
04/09/2026 17:35:58 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/datasets/DaniilOr/SemEval-2026-Task13/revision/df2aec18238a47fabfb22c79570db15fd1588aa3 "HTTP/1.1 200 OK"
04/09/2026 17:35:58 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/df2aec18238a47fabfb22c79570db15fd1588aa3/.huggingface.yaml "HTTP/1.1 404 Not Found"
04/09/2026 17:35:58 - INFO - httpx - HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=DaniilOr/SemEval-2026-Task13 "HTTP/1.1 200 OK"
04/09/2026 17:35:58 - INFO

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

04/09/2026 17:36:01 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/datasets/DaniilOr/SemEval-2026-Task13/resolve/df2aec18238a47fabfb22c79570db15fd1588aa3/task_a/task_a_validation_set.parquet "HTTP/1.1 302 Found"


...

Tokenising train:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenising val:   0%|          | 0/100000 [00:00<?, ? examples/s]

04/09/2026 17:40:39 - INFO - __main__ - Loss: FocalLossWithSmoothing (alpha=1, gamma=2, smoothing=0.05)
04/09/2026 17:40:40 - INFO - __main__ - ***** Training Configuration *****
04/09/2026 17:40:40 - INFO - __main__ -   Backbone           : microsoft/unixcoder-base
04/09/2026 17:40:40 - INFO - __main__ -   Max epochs         : 1
04/09/2026 17:40:40 - INFO - __main__ -   Batch size         : 16
04/09/2026 17:40:40 - INFO - __main__ -   Learning rate      : 2e-05
04/09/2026 17:40:40 - INFO - __main__ -   LR schedule        : cosine_with_restarts
04/09/2026 17:40:40 - INFO - __main__ -   Warmup steps       : 1875 / 31250
04/09/2026 17:40:40 - INFO - __main__ -   LLRD               : True (decay=0.9)
04/09/2026 17:40:40 - INFO - __main__ -   Mixed precision    : fp16=False, bf16=False
04/09/2026 17:40:40 - INFO - __main__ -   Augmentation       : True (p=0.25)
04/09/2026 17:40:40 - INFO - __main__ -   Multi-sample drop  : True
04/09/2026 17:40:40 - INFO - __main__ -   Gradient checkpoint:

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [18]:
from huggingface_hub import HfApi

try:
    # Automatically get the username of the authenticated user
    
    # Define repository name
    repo_name = f"dzungpham/SLA-SemEval-challenge"
    
    api = HfApi()
    logger.info(f"Creating/checking repo: {repo_name}")
    api.create_repo(repo_id=repo_name, repo_type="model", exist_ok=True)
    
    logger.info(f"Uploading {OUTPUT_DIR} to Hugging Face Hub...")
    # This uploads everything in the OUTPUT_DIR (checkpoints, best model, etc.)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=repo_name,
        repo_type="model",
        commit_message="Upload training checkpoints and best model"
    )
    logger.info(f"Upload complete! Model is available at: https://huggingface.co/{repo_name}")
except Exception as e:
    logger.error(f"Failed to upload to Hugging Face: {e}")
    logger.info("Ensure you are authenticated with write access and have set your HF_TOKEN correctly.")

04/09/2026 18:11:33 - INFO - __main__ - Creating/checking repo: dzungpham/SLA-SemEval-challenge
04/09/2026 18:11:33 - INFO - httpx - HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 409 Conflict"
04/09/2026 18:11:33 - INFO - __main__ - Uploading taskA-graphcodebert-focal to Hugging Face Hub...
No files have been modified since last commit. Skipping to prevent empty commit.
04/09/2026 18:11:33 - WARNING - huggingface_hub.hf_api - No files have been modified since last commit. Skipping to prevent empty commit.
04/09/2026 18:11:33 - INFO - httpx - HTTP Request: GET https://huggingface.co/api/models/dzungpham/SLA-SemEval-challenge/revision/main "HTTP/1.1 200 OK"
04/09/2026 18:11:33 - INFO - __main__ - Upload complete! Model is available at: https://huggingface.co/dzungpham/SLA-SemEval-challenge


In [ ]:
from itertools import chain
from tqdm.auto import tqdm
import re

def preprocess_code(code_str):
    """Preprocess code string."""
    code_str = re.sub(r'[\r\n]+', '\n', code_str)
    code_str = re.sub(r'[ \t]+', ' ', code_str)
    return code_str

@torch.no_grad()
def predict_on_test(checkpoint_dir, output_path, max_length=512, batch_size=16):
    """
    Load model from checkpoint and perform predictions on test set.

    Args:
        checkpoint_dir: Path to the checkpoint directory containing model files
        output_path: Path to save predictions CSV
        max_length: Maximum sequence length for tokenization
        batch_size: Batch size for inference
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    logger.info(f"Loading model and tokenizer from: {checkpoint_dir}")

    # Load model and tokenizer from checkpoint
    try:
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    except Exception as e:
        logger.error(f"Error loading checkpoint from {checkpoint_dir}: {e}")
        raise

    model = model.to(device)
    model.eval()
    logger.info(f"Model loaded on device: {device}")

    # Load test dataset from Hugging Face Hub
    logger.info("Loading test dataset...")
    ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test", streaming=True)
    it = iter(ds)
    first = next(it)
    stream = chain([first], it)

    def batcher(iterator, bs):
        """Helper function to create batches from iterator."""
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    # Perform inference on test set
    with open(output_path, "w") as f:
        f.write("id,label\n")
        logger.info("Predicting on test set...")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            # Preprocess codes
            codes = [preprocess_code(row["code"]) for row in batch]
            ids = [row["id"] for row in batch]

            # Tokenize
            enc = tokenizer(
                codes,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            # Get predictions
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()

            # Write to CSV
            for ex_id, pred in zip(ids, pred_labels):
                f.write(f"{ex_id},{pred}\n")

    logger.info(f"Predictions saved to {output_path}")


# ============== Prediction Configuration ==============
# Choose which checkpoint to use for prediction:
# Option 1: Use the best model checkpoint (highest macro F1)
CHECKPOINT_PATH = "taskA-unixcoder-focal/best_model"

# Option 2: Use the latest checkpoint
# CHECKPOINT_PATH = "taskA-unixcoder-focal/checkpoint-<step>"

OUT_CSV = "submission.csv"

# Run prediction using checkpoint
predict_on_test(
    checkpoint_dir=CHECKPOINT_PATH,
    output_path=OUT_CSV,
    max_length=512,
    batch_size=32,
)

logger.info(f"Predictions written to: {OUT_CSV}")


In [ ]:
import os
import zipfile
from pathlib import Path


def find_result_folders(base_path: Path, pattern_name: str) -> list[Path]:
    return [p for p in base_path.glob(pattern_name) if p.is_dir()]


def zip_folder(folder_path: Path, output_base_path: Path) -> bool:
    folder_name = folder_path.name
    zip_path = output_base_path / f"{folder_name}.zip"
    try:
        print(f"   -> Zipping folder: {folder_name}...")
        with zipfile.ZipFile(
            zip_path, mode="w", compression=zipfile.ZIP_DEFLATED
        ) as zipf:
            for file_path in folder_path.rglob("*"):
                if file_path.is_file():
                    arcname = file_path.relative_to(folder_path.parent)
                    zipf.write(file_path, arcname)
        print(f"  Created ZIP: {zip_path.name}")
        return True
    except Exception as exc:
        print(f"   Failed to zip {folder_name}: {exc}")
        return False


def zip_stats_results_folders(output_base_path: str, pattern_name: str) -> None:
    base = Path(output_base_path)
    base.mkdir(parents=True, exist_ok=True)
    result_folders = find_result_folders(base, pattern_name)
    if not result_folders:
        print(f"⚠️ No folders matching {pattern_name} found in '{output_base_path}'.")
        return
    print(f"Found {len(result_folders)} result folder(s) to zip.")
    successful = sum(1 for folder in result_folders if zip_folder(folder, base))
    print(
        f"\nDONE! Successfully zipped {successful} out of {len(result_folders)} folder(s)."
    )


if __name__ == "__main__":
    try:
        print("📦 Output Path:", OUTPUT_PATH)
        pattern_names = ["*checkpoint*"]
        output_root = os.getenv("OUTPUT_PATH") or globals().get("OUTPUT_PATH")
        if not output_root:
            raise ValueError("OUTPUT_PATH not defined")
        for pattern_name in pattern_names:
            zip_stats_results_folders(
                output_base_path=OUTPUT_PATH, pattern_name=pattern_name
            )
    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [ ]:
if is_colab:
    from google.colab import files
    files.download("/content/submission.csv")
from IPython.display import FileLink
FileLink("submission.csv")